# Final model on the full data set: fit, predict, save, and investigate the features

This notebook runs the **final model** with already tuned hyperparameters and writes the prediction files. It does **no** hyperparameter search. The steps:

1. set the paths and devices (one cell);
2. check the inputs (tuned `hyperparameters.json`, embeddings of `train`, `test_bulk`, `test_sc`);
3. refit the whole chain on **all** training samples and predict: Fisher sequence filter (if the tuned configuration uses one) → FAISS k-means clusters (centroids + radii) → cluster-occupancy features → class-aware prevalence filter → XGBoost;
4. validate and save the predictions (`predictions_test_bulk.tsv`, `predictions_test_sc.tsv`, optionally a `test_sc` booster without `log_depth`);
5. investigate which features drive the model: SHAP importance of clusters, covariates, the clusters' sequences, the k-mers chosen by the Fisher filter, and the explanation of single samples.

**Prerequisites.** Kernel = the conda environment `sceptr_env`. `data/embeddings/<method>/{train,test_bulk,test_sc}` must exist (made by `scripts/run_embeddings.sh` or the generators), and `hyperparameters.json` must come from `scripts/tune_hyperparameters.py` / `scripts/run_tuning.sh` for the **same** embedding method. The fit uses the same code as `run_pipeline.py --stage fit_predict`.

> **Not measured at full scale.** This notebook was tested on the tiny tutorial subset only. On the full data the Fisher step pools about 25 million training clonotypes (likely 10-20 GB of RAM) and the final fit, the SHAP step may take a long time. Run the first full-data attempt on a machine with plenty of RAM and watch it.

## Step 1. Settings

In [ ]:
import contextlib, io, json, os, sys, time, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

warnings.filterwarnings("ignore")
pd.set_option("display.width", 170)
pd.set_option("display.max_columns", 40)

def find_root(start=Path.cwd()):
    for d in [start, *start.parents]:
        if (d / "run_pipeline.py").exists():
            return d
    raise RuntimeError("run this notebook from inside the project (run_pipeline.py not found)")

ROOT = find_root()
sys.path.insert(0, str(ROOT))

# ---- paths: edit these ------------------------------------------------------------------------
DATA_DIR   = ROOT / "data"                                  # contains embeddings/<method>/{train,test_bulk,test_sc}
HP_PATH    = ROOT / "results" / "run" / "tuning" / "sceptr_kmer_kmeans" / "hyperparameters.json"   # the tuned configuration
OUTPUT_DIR = ROOT / "results" / "final"                     # predictions and importance tables are written here
MODELS_DIR = ROOT / "models" / "final"                      # fitted cluster model + booster are saved here

# ---- devices and options ------------------------------------------------------------------------
XGB_DEVICE    = "cpu"      # "cuda" on the GPU server (XGBoost)
KMEANS_DEVICE = "cpu"      # FAISS; "cuda" only if your FAISS build has kernels for your GPU (the PyPI faiss-gpu-cu12 wheel works on a V100)
N_JOBS = -1                # CPU workers (-1 = all cores)
SEED = None                # None = the seed stored in the hyperparameters (recommended)
SC_WITHOUT_LOG_DEPTH = True    # also write test_sc predictions from a booster refit without log_depth
                               # (log_depth differs ~30x between bulk and single-cell repertoires)
# true labels of the test sets, for the confusion matrices of Step 4. They are NOT in data/raw (the test sets are unlabelled
# there): give a TSV/CSV per set with the columns sample_id and disease_group (autoimmunity | cancer | infection), or None.
TEST_LABELS = {"test_bulk": None, "test_sc": None}      # e.g. {"test_bulk": ROOT / "data" / "test_labels" / "test_bulk.tsv", ...}
RUN_LODO = False           # Step 6: leave-one-disease-out validation. Refits the whole chain once per held-out disease: slow on the full data
LODO_DISEASES = None       # None = every training disease; or a list of diseases to hold out, e.g. ["Melanoma", "HIV"], to bound the run time
EXCLUDE_DISEASES = None    # None = the diseases excluded in tuning (stored in hyperparameters.json); a list overrides it (warns)

import faiss, xgboost
from src import pipeline as pl, embeddings as emb, evaluation as ev, model as mdl, clustering as cl, feature_selection as fs
print("python:", sys.executable)
print(f"faiss {faiss.__version__} | xgboost {xgboost.__version__}")
if "sceptr_env" not in sys.executable:
    print("WARNING: this kernel is not the sceptr_env conda environment")

## Step 2. Check the inputs

The tuned configuration is read from `hyperparameters.json`. Everything the final fit needs is in it: the embedding method, the Fisher filter settings (`filters`), the clustering (`clustering`: K and radius quantile for k-means), the XGBoost parameters with their number of boosting rounds, the prevalence-filter settings, the excluded diseases and the seed. **The Fisher filter is not switched off**: if the tuning used `seq_filter = kmer`, the final fit uses it too.

In [ ]:
if not Path(HP_PATH).exists():
    raise FileNotFoundError(f"no hyperparameters at {HP_PATH}: run the tuning first, or edit HP_PATH")
hp = pl.load_hyperparameters(HP_PATH)
METHOD = hp.get("embedding_method", "sceptr")      # embedding folder: sceptr, or sceptr_<variant>
SEED = hp.get("random_seed", 42) if SEED is None else SEED
cparams = pl.cluster_params_of(hp["clustering"])
filters = {**pl.DEFAULT_FILTERS, **hp.get("filters", {}), "seq_filter": hp["seq_filter"]}

print(f"embedding method : {METHOD}")
print(f"clustering       : FAISS k-means {cparams}")
print(f"sequence filter  : {filters['seq_filter']}" + ("" if filters["seq_filter"] == "none" else
      f" (k={filters['kmer_k']}, with_v={filters['with_v']}, trim {filters['kmer_trim_start']}+{filters['kmer_trim_end']}, "
      f"{filters['fisher_select']}: {filters[pl.STRENGTH_KEY[filters['fisher_select']]]}, min_samples={filters['min_samples']})"))
print(f"prevalence filter: {filters['prevalence_mode']} (min_prevalence={filters['min_prevalence']}, min count {filters['min_prevalence_count']})")
print(f"XGBoost          : {hp['xgboost']}")
print(f"excluded diseases: {hp.get('exclude_diseases') or 'none'} | seed {SEED} | tuned on {hp.get('n_train_samples')} samples")

missing = [ds for ds in ("train", "test_bulk", "test_sc")
           if not (Path(emb.embedding_dir(DATA_DIR, METHOD, ds)) / "embedding_config.json").exists()]
if missing:
    raise FileNotFoundError(f"no {METHOD} embeddings for {missing} under {DATA_DIR / 'embeddings' / METHOD}")
pl.check_device(XGB_DEVICE)
cl.check_available(KMEANS_DEVICE)     # fails clearly if FAISS (or a usable FAISS GPU) is missing
print("\ninputs OK")

## Step 3. Fit on all training samples and predict

`pl.fit_predict` does the whole chain once, on **all** training samples (no folds):

1. the Fisher sequence filter is fitted on all training samples (if the configuration has one);
2. the clustering is fitted on the surviving training sequences; every sequence of every sample, including the test samples, is then assigned to its cluster or left *unassigned* (k-means: beyond the cluster's radius);
3. per-sample cluster-occupancy features plus `unassigned`, `evenness` and `log_depth`;
4. the class-aware prevalence filter (fitted on the training samples), the same columns applied to the test samples;
5. the XGBoost booster with the tuned parameters and the fixed number of rounds, balanced sample weights;
6. predictions for `test_bulk` and `test_sc` (validated against each dataset's `metadata.tsv`), the SHAP cluster importance table, and the saved cluster model and booster.

Files written to `OUTPUT_DIR`: `predictions_test_bulk.tsv`, `predictions_test_sc.tsv` (and `predictions_test_sc_without_log_depth.tsv`), `cluster_importance.csv`, `prevalence_report.csv`, `sequence_filter_report.csv`; to `MODELS_DIR`: `cluster_model.pkl`, `booster.json`, `model_config.json`.

In [ ]:
ctx = pl.load_context(DATA_DIR, METHOD, seed=SEED, n_jobs=N_JOBS, xgboost_device=XGB_DEVICE,
                      exclude_diseases=EXCLUDE_DISEASES if EXCLUDE_DISEASES is not None else hp.get("exclude_diseases"),
                      kmeans_device=KMEANS_DEVICE)
meta = ctx["meta"]
print(f"{len(meta)} training samples:", meta["disease_group"].value_counts().to_dict())
print("test sets:", {k: len(v["meta"]) for k, v in ctx["test"].items()})

In [ ]:
OUTPUT_DIR, MODELS_DIR = Path(OUTPUT_DIR), Path(MODELS_DIR)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

t0 = time.time()
r, booster, imp = pl.fit_predict(ctx, hp, str(OUTPUT_DIR), str(MODELS_DIR), features_dir=str(DATA_DIR / "features" / METHOD),
                                 shap_plots=False, sc_without_log_depth=SC_WITHOUT_LOG_DEPTH)
print(f"\nfinal fit + prediction done in {(time.time() - t0) / 60:.1f} min")

## Step 4. Check and save the predictions

`write_predictions` already validated each file (all expected sample ids, no NaN, rows sum to 1, fixed class order). Here we read the saved files back, look at them, and summarise the predicted classes. **These tab-separated files are the final result**; the summary table is saved as well.

In [ ]:
files = sorted(OUTPUT_DIR.glob("predictions_*.tsv"))
rows = []
for f in files:
    df = pd.read_csv(f, sep="\t")
    name = f.stem.replace("predictions_", "")
    base = name.replace("_without_log_depth", "")
    expected = ctx["test"][base]["meta"]["sample_id"].tolist()
    ok = (set(df["sample_id"]) == set(expected)) and len(df) == len(expected) and np.allclose(df[emb.CLASS_NAMES].sum(axis=1), 1, atol=1e-4)
    pred = df[emb.CLASS_NAMES].idxmax(axis=1).value_counts().reindex(emb.CLASS_NAMES, fill_value=0)
    rows.append({"file": f.name, "samples": len(df), "ids match metadata & rows sum to 1": ok,
                 **{f"predicted {c}": int(pred[c]) for c in emb.CLASS_NAMES},
                 "mean max probability": round(float(df[emb.CLASS_NAMES].max(axis=1).mean()), 3)})
summary = pd.DataFrame(rows)
summary.to_csv(OUTPUT_DIR / "predictions_summary.csv", index=False)
display(summary)
display(pd.read_csv(OUTPUT_DIR / "predictions_test_bulk.tsv", sep="\t").head())

In [ ]:
# distribution of the predicted probabilities: a model with no signal stays close to 1/3 everywhere
fig, axes = plt.subplots(1, len([f for f in files if "without" not in f.name]), figsize=(11, 3.4), squeeze=False)
for ax, f in zip(axes[0], [f for f in files if "without" not in f.name]):
    df = pd.read_csv(f, sep="\t")
    for c in emb.CLASS_NAMES:
        ax.hist(df[c], bins=20, range=(0, 1), alpha=0.5, label=c)
    ax.set_title(f.stem.replace("predictions_", "")); ax.set_xlabel("predicted probability"); ax.legend()
plt.tight_layout(); plt.show()

### 4b. Confusion matrices for `test_bulk` and `test_sc`

A confusion matrix compares the predicted class with the **true** class, and the test sets in `data/raw` have no labels (only sample ids). Set `TEST_LABELS` in Step 1 to a table with the columns `sample_id` and `disease_group` for each test set you have labels for. Sets without labels are skipped with a message; nothing is plotted from guessed labels.

Rows are the true class and columns the predicted class (argmax of the probabilities); each cell shows the count and the row-normalised share. For `test_sc` the matrix is drawn for both prediction files (with and without `log_depth`) when both exist. The figure titles carry accuracy, balanced accuracy and macro ROC-AUC. Figures and count tables are saved to `OUTPUT_DIR`.

In [ ]:
from sklearn.metrics import confusion_matrix

def load_test_labels(path, ids):
    """Class index per sample id (in the order of ``ids``) from a TSV/CSV with sample_id and disease_group."""
    df = pd.read_csv(path, sep=None, engine="python", dtype=str)
    missing = {"sample_id", "disease_group"} - set(df.columns)
    if missing:
        raise ValueError(f"{path}: missing column(s) {sorted(missing)}; found {list(df.columns)}")
    unknown = set(df["disease_group"]) - set(emb.LABEL_MAP)
    if unknown:
        raise ValueError(f"{path}: unknown disease_group values {sorted(unknown)}; expected {emb.CLASS_NAMES}")
    lab = df.drop_duplicates("sample_id").set_index("sample_id")["disease_group"]
    absent = [i for i in ids if i not in lab.index]
    if absent:
        raise ValueError(f"{path}: no label for {len(absent)} of {len(ids)} samples, e.g. {absent[:5]}")
    return lab.loc[ids].map(emb.LABEL_MAP).to_numpy(dtype=int)

def plot_confusion(ax, y_true, P, title):
    y_pred = P.argmax(axis=1)
    cm = confusion_matrix(y_true, y_pred, labels=range(3))
    norm = cm / np.maximum(cm.sum(axis=1, keepdims=True), 1)
    ax.imshow(norm, vmin=0, vmax=1, cmap="Blues")
    for i in range(3):
        for j in range(3):
            ax.text(j, i, f"{cm[i, j]}\n({norm[i, j]:.0%})", ha="center", va="center",
                    color="white" if norm[i, j] > 0.5 else "black")
    ax.set_xticks(range(3)); ax.set_xticklabels(emb.CLASS_NAMES, rotation=20)
    ax.set_yticks(range(3)); ax.set_yticklabels(emb.CLASS_NAMES)
    ax.set_xlabel("predicted"); ax.set_ylabel("true"); ax.set_title(title, fontsize=10)
    return cm

drawn = False
for ds in ("test_bulk", "test_sc"):
    path = TEST_LABELS.get(ds)
    if path is None:
        print(f"{ds}: no labels given (TEST_LABELS) - confusion matrix skipped")
        continue
    variants = [(f"predictions_{ds}.tsv", ds)]
    if ds == "test_sc":
        variants.append(("predictions_test_sc_without_log_depth.tsv", "test_sc without log_depth"))
    variants = [(f, name) for f, name in variants if (OUTPUT_DIR / f).exists()]
    fig, axes = plt.subplots(1, len(variants), figsize=(5.2 * len(variants), 4.6), squeeze=False)
    for ax, (fname, name) in zip(axes[0], variants):
        pred = pd.read_csv(OUTPUT_DIR / fname, sep="\t")
        y_true = load_test_labels(path, pred["sample_id"].tolist())
        P = pred[emb.CLASS_NAMES].to_numpy()
        try:
            m = ev.compute_metrics(y_true, P)
            stats = (f"acc {np.mean(P.argmax(1) == y_true):.2f} | bal. acc {m['balanced_accuracy']:.2f} | "
                     f"macro AUC {m['macro_roc_auc_pooled']:.2f}")
        except Exception as e:                       # e.g. a class with no sample among the labels
            stats = f"acc {np.mean(P.argmax(1) == y_true):.2f} (AUC not computable: {type(e).__name__})"
        cm = plot_confusion(ax, y_true, P, f"{name} (n={len(y_true)})\n{stats}")
        pd.DataFrame(cm, index=[f"true {c}" for c in emb.CLASS_NAMES], columns=[f"pred {c}" for c in emb.CLASS_NAMES]) \
            .to_csv(OUTPUT_DIR / f"confusion_matrix_{name.replace(' ', '_')}.csv")
    plt.tight_layout()
    fig.savefig(OUTPUT_DIR / f"confusion_matrix_{ds}.png", dpi=130)
    plt.show()
    drawn = True
if not drawn:
    print("\nNo labels configured: set TEST_LABELS in Step 1 to plot confusion matrices.")

## Step 5. Feature importance investigation

The features of the model are the **cluster-occupancy proportions** (`c0`, `c1`, ...: the share of a sample's distinct clonotypes that fall in that cluster) plus three repertoire-level covariates (`unassigned`, `evenness`, `log_depth`). Importance is computed with exact TreeSHAP (XGBoost's `pred_contribs`) on the **training samples with balanced weights**, per class, on the log-odds scale; `importance_overall` is the sum over the three classes. Direction columns are the Spearman correlation between a feature's value and its SHAP value for that class (positive: more occupancy pushes towards the class).

SHAP describes what *this model* uses, not causal biology: a cluster that identifies one study or one disease ranks high too. The sections below therefore also check how concentrated each important cluster is in a single training disease.

### 5a. Which features matter

In [ ]:
imp = pd.read_csv(OUTPUT_DIR / "cluster_importance.csv")
print(f"{(imp['feature_type'] == 'cluster').sum()} cluster features + {(imp['feature_type'] == 'extra').sum()} covariates; top 15:")
display(imp.head(15).round(4))
cov = imp.loc[imp["feature_type"] == "extra", "importance_overall"].sum() / imp["importance_overall"].sum()
print(f"share of the total importance carried by the covariates (unassigned / evenness / log_depth): {cov:.0%}")
print("If this share is large, the model partly reads repertoire size or diversity rather than sequence content.")
print("importance of the covariates:")
display(imp[imp["feature_type"] == "extra"].round(4))

In [ ]:
# stacked bar plot of the top features: mean |SHAP| per class
top = imp.head(20).iloc[::-1]
fig, ax = plt.subplots(figsize=(8, 6))
left = np.zeros(len(top))
for c in emb.CLASS_NAMES:
    ax.barh(top["feature"], top[f"importance_{c}"], left=left, label=c)
    left += top[f"importance_{c}"].to_numpy()
ax.set_xlabel("mean |SHAP| (balanced training samples), stacked by class"); ax.legend(loc="lower right")
ax.set_title("Top 20 features")
plt.tight_layout(); plt.savefig(OUTPUT_DIR / "shap_bar.png", dpi=130); plt.show()

### 5b. Direction of the effect (beeswarm)

A dot is a training sample; x = the feature's effect on that class's log-odds; colour = the feature's value (red = high occupancy). Red dots on the right: more sequences in this cluster push a sample towards the class. At most 1,500 randomly chosen training samples are drawn.

In [ ]:
import shap
names = r["names"]
X_tr = r["X_train"]
rng = np.random.default_rng(SEED)
idx = rng.choice(len(X_tr), size=min(1500, len(X_tr)), replace=False)
phi, bias = ev.shap_contributions(booster, X_tr[idx], names)        # (samples, classes, features)
print("additivity check (max |contributions + bias - raw output|):", f"{ev.check_additivity(booster, X_tr[idx], names):.2e}")
for c, cname in enumerate(emb.CLASS_NAMES):
    plt.figure()
    shap.summary_plot(phi[:, c, :], X_tr[idx], feature_names=names, max_display=12, show=False)
    plt.title(f"SHAP beeswarm: effect on '{cname}' (training samples)")
    plt.tight_layout(); plt.savefig(OUTPUT_DIR / f"shap_beeswarm_{cname}.png", dpi=130, bbox_inches="tight"); plt.show()

### 5c. What is inside the important clusters

For the highest-ranked clusters: how many training sequences they contain, the V/J genes and CDR3 lengths, the cluster's radius (k-means), in how many training samples the cluster is occupied, and which single training disease contributes the largest share of the cluster's occupancy compared with what its sample count alone would give ("expected"). A cluster dominated by one disease (ratio >= 2) is a likely **study marker** rather than a disease signature; one V gene in ~100 % of a small cluster is a similar red flag.

In [ ]:
model = r["model"]
f_tr = r["f_train"]
train_ids, dis_train = meta["sample_id"].to_numpy(), meta["disease"].to_numpy()
dis_tr = pd.Series(dis_train, index=f_tr.index)
n_per_disease = dis_tr.value_counts()

def cluster_members(model, label):
    keys = model["keys_sorted"][model["labels_sorted"] == label]
    parts = pd.Series([k.decode() for k in keys]).str.split("+", n=1, expand=True)
    vj = parts[1].str.split("_", n=1, expand=True)
    return pd.DataFrame({"cdr3": parts[0], "v": vj[0], "j": vj[1]})

def cluster_radius(model, label):
    """Radius of the original centroid that became cluster `label`."""
    return float(model["radius"][np.where(model["remap"] == label)[0][0]])

N_INSPECT = 10
rows = []
for _, row in imp[imp["feature_type"] == "cluster"].head(N_INSPECT).iterrows():
    label = int(row["feature"][1:])
    m = cluster_members(model, label)
    mass = f_tr[row["feature"]].groupby(dis_tr).sum()
    top_dis = mass.idxmax()
    share, expected = mass.max() / mass.sum(), n_per_disease[top_dis] / len(dis_tr)
    rows.append({"cluster": row["feature"], "importance": round(row["importance_overall"], 3), "top class": row["top_class"],
                 "members": len(m), "radius": round(cluster_radius(model, label), 3),
                 "median CDR3 length": int(m["cdr3"].str.len().median()),
                 "top V (share)": f"{m['v'].value_counts().index[0]} ({m['v'].value_counts(normalize=True).iloc[0]:.0%})",
                 "top J (share)": f"{m['j'].value_counts().index[0]} ({m['j'].value_counts(normalize=True).iloc[0]:.0%})",
                 "samples occupied": f"{int((f_tr[row['feature']] > 0).sum())}/{len(f_tr)}",
                 "top disease (share vs expected)": f"{top_dis} {share:.0%} vs {expected:.0%}",
                 "concentration ratio": round(share / expected, 2),
                 "example CDR3s": ", ".join(m["cdr3"].head(3))})
cluster_table = pd.DataFrame(rows)
cluster_table.to_csv(OUTPUT_DIR / "top_cluster_summary.csv", index=False)
display(cluster_table)
flag = cluster_table.loc[cluster_table["concentration ratio"] >= 2, "cluster"].tolist()
print("clusters whose occupancy is >= 2x concentrated in one training disease (possible study marker):", flag or "none")

### 5d. The sequences of a cluster

Importance is a property of the **cluster**, so the sequences behind an important cluster are simply its members. `cluster_members(model, label)` (Step 5c) returns the CDR3, V and J of every training sequence the k-means model assigned to a cluster. The cell lists them for the most important cluster and saves them to `OUTPUT_DIR` for motif or database checks. Change `CLUSTER` to look at another one.

In [ ]:
CLUSTER = imp[imp["feature_type"] == "cluster"].iloc[0]["feature"]        # e.g. "c12": the most important cluster
members = cluster_members(model, int(CLUSTER[1:]))
members.to_csv(OUTPUT_DIR / f"cluster_{CLUSTER}_members.csv", index=False)
print(f"{CLUSTER}: {len(members):,} member sequences, saved to {OUTPUT_DIR / f'cluster_{CLUSTER}_members.csv'}")
display(members.head(10))
print("CDR3 length distribution:")
display(members["cdr3"].str.len().value_counts().sort_index().to_frame("sequences").T)
print("most frequent V genes:", members["v"].value_counts().head(5).to_dict())

### 5e. What the Fisher filter selected

If the tuned configuration uses the Fisher sequence filter, the selected features (here CDR3 k-mers) are the first thing that shapes the clusters: only training sequences carrying a selected k-mer enter the clustering. The report lists the selected features per class with their Fisher p, BH-adjusted q and odds ratio (a feature with q far above 0.05 passed only because of the percentile / top-k selection mode). The prevalence report shows which cluster columns were kept and which were kept *only* by the class-aware rule.

In [ ]:
rep_path = OUTPUT_DIR / "sequence_filter_report.csv"
if rep_path.exists():
    rep = pd.read_csv(rep_path)
    rep["class"] = rep["class"].map(dict(enumerate(emb.CLASS_NAMES)))
    print(f"{len(rep):,} selected features. Per class, the 5 with the smallest p:")
    display(rep.sort_values(["class", "p"]).groupby("class").head(5).round(4).reset_index(drop=True))
    print("selected features per class:", rep["class"].value_counts().to_dict(),
          "| median q:", round(float(rep["q"].median()), 3))
else:
    print("no sequence filter in this configuration")

prev = pd.read_csv(OUTPUT_DIR / "prevalence_report.csv")
print(f"\ncluster columns kept by the prevalence filter: {len(prev)}; kept only by the class-aware rule: {int(prev['only_class_aware'].sum())}")

### 5f. Why one sample got its prediction

For a single test sample, the features that pushed the prediction most, for its predicted class (SHAP contribution on the log-odds scale; positive = towards the class). Change `SAMPLE` to any `test_bulk` or `test_sc` id.

In [ ]:
DATASET = "test_bulk"
ids = ctx["test"][DATASET]["meta"]["sample_id"].tolist()
SAMPLE = ids[0]
pos = ids.index(SAMPLE)
X_s = r["X_evals"][DATASET][pos:pos + 1]
phi_s, bias_s = ev.shap_contributions(booster, X_s, names)
proba = mdl.predict_proba(booster, X_s, names)[0]
c = int(np.argmax(proba))
print(f"{DATASET} / {SAMPLE}: predicted {emb.CLASS_NAMES[c]}  (probabilities: "
      f"{{k: round(float(v), 3) for k, v in zip(emb.CLASS_NAMES, proba)}})")
contrib = pd.DataFrame({"feature": names, "value": X_s[0], "SHAP (towards predicted class)": phi_s[0, c, :]})
contrib = contrib.reindex(contrib["SHAP (towards predicted class)"].abs().sort_values(ascending=False).index)
display(contrib.head(10).round(4).reset_index(drop=True))

## Step 6. Leave-one-disease-out (LODO) validation

Stratified CV (used in tuning) hides samples at random, so training and validation samples can come from the **same study**. If a disease is separable from the others mainly through a study or batch fingerprint, CV looks good although the model has learned the study, not the disease. LODO asks the harder question: **can the model call a disease it has never seen?** For every training disease, all its samples are held out, the whole chain is refit on the remaining diseases with the **tuned configuration and no search** (Fisher filter, FAISS k-means, prevalence filter, XGBoost with inner early stopping), and the held-out samples are predicted.

Reading it:
* each held-out disease belongs to **one** class, so a per-disease AUC is undefined; the macro ROC-AUC is computed on the **pooled** LODO predictions;
* per disease: accuracy (share of its samples predicted as the right class) and the mean probability given to the true class; diseases with fewer than 10 samples are flagged as noisy;
* compare the pooled LODO macro AUC with the stratified CV macro AUC of the tuning run: **a large drop means much of the CV score came from recognising diseases or studies**, not from signal that transfers to unseen diseases. A disease with high accuracy in LODO is the more convincing evidence;
* LODO needs a class to stay represented in training: a class with a single disease cannot be held out and still be learned, so its fold is naturally hard.

**Cost.** One full chain per disease: with ~19 diseases that is about 19 times a final fit (without the test-set prediction). I have not timed it on the full data. Set `RUN_LODO = True` in Step 1 to run it, and use `LODO_DISEASES` to hold out only some diseases (the others stay in the training sets). Results go to `OUTPUT_DIR / "lodo"`.

In [ ]:
if not RUN_LODO:
    print("RUN_LODO is False: LODO skipped (set RUN_LODO = True in Step 1; it refits the whole chain once per held-out disease)")
else:
    params = mdl.xgb_params_from(hp["xgboost"], XGB_DEVICE, SEED)
    t0 = time.time()
    lodo_m, lodo_per, lodo_oof = pl.run_lodo(ctx, cparams, filters, params, only_diseases=LODO_DISEASES)
    print(f"\nLODO done in {(time.time() - t0) / 60:.1f} min")
    lodo_dir = OUTPUT_DIR / "lodo"
    ev.write_metrics(lodo_dir, lodo_m)
    lodo_per.to_csv(lodo_dir / "per_disease.csv", index=False)
    lodo_oof.to_csv(lodo_dir / "oof_predictions.tsv", sep="\t", index=False)

In [ ]:
if RUN_LODO:
    cv_file = Path(HP_PATH).parent / "cv_metrics.json"
    cv_auc = json.loads(cv_file.read_text())["macro_roc_auc_pooled"] if cv_file.exists() else None
    print(f"pooled LODO: macro ROC-AUC {lodo_m['macro_roc_auc_pooled']:.3f}, macro F1 {lodo_m['f1_macro']:.3f}, "
          f"balanced accuracy {lodo_m['balanced_accuracy']:.3f}  ({lodo_m['n_samples']} samples held out)")
    if cv_auc is not None:
        print(f"stratified CV of the tuning run: macro ROC-AUC {cv_auc:.3f}  ->  LODO minus CV = {lodo_m['macro_roc_auc_pooled'] - cv_auc:+.3f}")
    else:
        print(f"(no {cv_file.name} next to the hyperparameters: cannot compare with the stratified CV)")
    per = lodo_per.sort_values("accuracy", ascending=False).reset_index(drop=True)
    display(per.round(3))
    print("diseases flagged flag_small_n=True (fewer than 10 samples) give noisy accuracies")

    colors = dict(zip(emb.CLASS_NAMES, ["#4c78a8", "#e45756", "#54a24b"]))
    fig, axes = plt.subplots(1, 2, figsize=(14, max(4, 0.33 * len(per) + 1.5)), gridspec_kw={"width_ratios": [1.5, 1]})
    ax = axes[0]
    ax.barh(per["disease"][::-1], per["accuracy"][::-1], color=[colors[c] for c in per["true_class"][::-1]])
    for yv, (acc, n) in enumerate(zip(per["accuracy"][::-1], per["n"][::-1])):
        ax.text(min(acc + 0.01, 0.97), yv, f"n={n}", va="center", fontsize=8)
    ax.axvline(1 / 3, color="k", ls=":", lw=1); ax.set_xlim(0, 1.05)
    ax.set_xlabel("share of held-out samples predicted as the right class"); ax.set_title("LODO accuracy per disease (dotted: 1/3)")
    ax.legend(handles=[plt.Rectangle((0, 0), 1, 1, color=c) for c in colors.values()], labels=list(colors), loc="lower right", fontsize=8)
    y_l = lodo_oof["true_class"].map(emb.LABEL_MAP).to_numpy()
    plot_confusion(axes[1], y_l, lodo_oof[emb.CLASS_NAMES].to_numpy(), f"pooled LODO predictions (n={len(y_l)})")
    plt.tight_layout(); fig.savefig(OUTPUT_DIR / "lodo" / "lodo_summary.png", dpi=130); plt.show()

## Step 7. Result files

Everything below is written by this notebook. The two prediction files are the deliverable; the other files document how the model reached them.

In [ ]:
for d in (OUTPUT_DIR, MODELS_DIR):
    print(d)
    for p in sorted(d.iterdir()):
        print(f"   {p.name:50s} {p.stat().st_size / 1024:10.1f} KB")
print("\nto reproduce the predictions from the saved model without refitting: load cluster_model.pkl and booster.json, "
      "assign each test sample's sequences to clusters (cl.extract_features), keep the columns listed in model_config.json['feature_names'].")